In [4]:
"""
Project ICICI_Close onto a normalized vector space of all other factors.

Price ≈ w1*Gold + w2*USD_INR + w3*... + R (residual)

Each column (including the target) is standardized to a z-score, so every
factor becomes a unit-scale axis in the vector space. Ridge regression finds
the weight vector w that best projects the target onto the factor axes.
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

# ---- 1. Loading & cleaning ----
df = pd.read_csv("/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/Model data/master_raw_aligned.csv", parse_dates=["Date"])
df = df.sort_values("Date").dropna().reset_index(drop=True) 
target_col = "ICICI_Close"
feature_cols = [c for c in df.columns if c not in ["Date", target_col]]

X_raw = df[feature_cols].values
y_raw = df[[target_col]].values

# ---- 2. Normalize everything to z-scores (build the vector space) ----
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X = x_scaler.fit_transform(X_raw)
y = y_scaler.fit_transform(y_raw).ravel()

# ---- 3. Fit Ridge with time-series cross-validation to pick alpha ----
tscv = TimeSeriesSplit(n_splits=5)
alphas = np.logspace(-3, 3, 50)
model = RidgeCV(alphas=alphas, cv=tscv)
model.fit(X, y)

print(f"Chosen alpha (regularization strength): {model.alpha_:.4f}")

# ---- 4. Weights (projection coefficients in the standardized vector space) ----
weights = pd.Series(model.coef_, index=feature_cols).sort_values(key=abs, ascending=False)
print("\nWeight vector (standardized units — directly comparable across factors):")
print(weights.round(4))

# ---- 5. Predictions, residuals, fit quality ----
y_pred_std = model.predict(X)
residuals_std = y - y_pred_std
r2 = r2_score(y, y_pred_std)
print(f"\nR^2 (in standardized space): {r2:.4f}")

# Convert predictions/residuals back to real rupee terms
y_pred_price = y_scaler.inverse_transform(y_pred_std.reshape(-1, 1)).ravel()
residual_price = df[target_col].values - y_pred_price

# ---- 6. Save outputs ----
out = df[["Date", target_col]].copy()
out["Predicted_Close"] = y_pred_price
out["Residual"] = residual_price
out.to_csv("/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/PHASES/Phase3/icici_projection_predictions.csv", index=False)

weights.to_csv("/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Qoin/PHASES/Phase3/icici_projection_weights.csv", header=["weight"])

print("\nSaved: icici_projection_predictions.csv, icici_projection_weights.csv")
print(f"Mean abs residual (₹): {np.mean(np.abs(residual_price)):.2f}")

Chosen alpha (regularization strength): 0.3728

Weight vector (standardized units — directly comparable across factors):
Nifty_Close                  0.4941
US_5Y                       -0.3695
US_10Y                       0.3282
BankNifty_Close              0.3134
WPI_Index                    0.2744
Bond_5Y                     -0.1744
CPI_Index                   -0.1383
NII (₹ Cr)                   0.1157
CRR_%                        0.1142
Repo_Rate_%                  0.0886
Bond_10Y                     0.0653
ICICI_RSI_14                 0.0421
Nifty_RSI_14                -0.0418
Net Profit (₹ Cr)            0.0412
Real_GDP_YoY_Growth_%       -0.0363
Trade_Balance_Billion_USD    0.0093
DII_Net                      0.0055
BankNifty_RSI_14            -0.0053
FII_Net                     -0.0050
USD_INR                     -0.0015
Gold_Close                  -0.0004
dtype: float64

R^2 (in standardized space): 0.9936

Saved: icici_projection_predictions.csv, icici_projection_weights.csv


In [10]:
print(y)

[-0.90722251 -0.91280222 -0.91740543 ...  1.96715565  1.93084128
  1.93493299]


In [3]:
"""
QOIN — Phase 3: Vector Space Visualization

Auto-selects the 3 highest-|weight| factors from the Johansen result and
plots them as literal spatial axes. Each trading day is one point in this
3-factor slice. Point color = how far the FULL residual (all 11 factors,
not just these 3) sits from its own mean that day -- i.e. the actual
mean-reversion state of the model. The black line is the Johansen weight
vector's direction, restricted to just these 3 dimensions -- the "axis"
your original idea describes, made literal.

IMPORTANT: this is a 3-of-11-dimensional SLICE, not the whole model. A
point that looks "off the line" here may be perfectly explained by the
other 8 factors moving that day. Don't read wobble in this picture alone
as the model failing -- the console output tells you what % of the total
weight mass these 3 axes actually represent.

Requires phase3_johansen.py in the SAME FOLDER -- this script reuses its
data loading, Johansen run, and residual logic so the two scripts can
never disagree with each other.

Run locally: python phase3_vector_space.py
Opens an interactive, rotatable 3D plot in your browser and also saves
phase3_vector_space.html so you can reopen it anytime without re-running.
"""

import numpy as np
import plotly.graph_objects as go

from johansen import (
    DATA_PATH,
    TARGET_COL,
    load_and_prepare,
    run_johansen,
    build_residual,
    adf_on_residual,
)


def pick_top3_factors(weights, factor_names):
    order = np.argsort(-np.abs(weights))[:3]
    top3 = [factor_names[i] for i in order]
    coverage = np.abs(weights[order]).sum() / np.abs(weights).sum()
    return top3, coverage


def make_figure(df, residual, weights, factor_cols, target_col):
    top3, coverage = pick_top3_factors(weights, factor_cols)
    print(f"Top 3 factors by |Johansen weight|: {top3}")
    print(f"These 3 account for {coverage:.1%} of total |weight| mass "
          f"across all {len(factor_cols)} factors -- the remaining "
          f"{1 - coverage:.1%} of the story lives outside this picture.\n")

    x, y, z = [df[f].values for f in top3]

    resid_z = (residual - residual.mean()) / residual.std()

    scatter = go.Scatter3d(
        x=x, y=y, z=z,
        mode="markers",
        marker=dict(
            size=3,
            color=resid_z,
            colorscale="RdBu",
            cmid=0,
            colorbar=dict(title="Residual<br>(SDs from mean)"),
        ),
        text=[f"{target_col}: {v:.1f}" for v in df[target_col]],
        hovertemplate=(
            f"{top3[0]}: %{{x:.2f}}<br>"
            f"{top3[1]}: %{{y:.2f}}<br>"
            f"{top3[2]}: %{{z:.2f}}<br>"
            "%{text}<extra></extra>"
        ),
        name="Trading days",
    )

    idx = [factor_cols.index(f) for f in top3]
    w3 = np.array([weights[i] for i in idx])
    w3 = w3 / np.linalg.norm(w3)

    centroid = np.array([x.mean(), y.mean(), z.mean()])
    spread = np.array([x.std(), y.std(), z.std()]).mean() * 2.5
    p0 = centroid - w3 * spread
    p1 = centroid + w3 * spread

    axis_line = go.Scatter3d(
        x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
        mode="lines",
        line=dict(color="black", width=6),
        name="Johansen axis (this slice)",
    )

    fig = go.Figure(data=[scatter, axis_line])
    fig.update_layout(
        title=(
            f"QOIN Phase 3 — Vector Space Slice<br>"
            f"<sup>Axes: {top3[0]}, {top3[1]}, {top3[2]} "
            f"({coverage:.0%} of total weight mass) | "
            f"color = full-model residual, SDs from mean</sup>"
        ),
        scene=dict(
            xaxis_title=top3[0],
            yaxis_title=top3[1],
            zaxis_title=top3[2],
        ),
        width=1000, height=800,
    )
    return fig


if __name__ == "__main__":
    df = load_and_prepare(DATA_PATH)
    factor_cols = [c for c in df.columns if c != TARGET_COL]

    result, rank = run_johansen(df)
    if rank == 0:
        raise SystemExit(
            "No cointegration found (rank 0) -- there is no valid Johansen "
            "weight vector to build axes from. Resolve that in "
            "phase3_johansen.py before visualizing."
        )

    residual, weights_full = build_residual(df, result, TARGET_COL)
    _, resid_pval = adf_on_residual(residual)
    if resid_pval >= 0.05:
        raise SystemExit(
            "Residual is not stationary (p >= 0.05) -- these weights are "
            "not trustworthy. Fix the basis before visualizing it."
        )

    col_list = df.columns.tolist()
    weights = np.array([weights_full[col_list.index(f)] for f in factor_cols])

    fig = make_figure(df, residual, weights, factor_cols, TARGET_COL)
    fig.write_html("phase3_vector_space.html")
    print("Saved interactive plot to phase3_vector_space.html")
    fig.show()

ModuleNotFoundError: No module named 'johansen'